# Temporal Fusion Transformer Training

This notebook trains a Temporal Fusion Transformer (TFT) for 14-day
daily demand forecasting using 60 days of historical context.

The model uses calendar features and planned promotion information
to support promotion-aware demand forecasting.

In [ ]:
from pytorch_forecasting.metrics import QuantileLoss
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
# Pandas is used to load our prepared time-series dataframe.
import pandas as pd

# PyTorch provides the underlying deep-learning functionality.
import torch

# PyTorch Forecasting provides the time-series dataset structure
# and the Temporal Fusion Transformer model.
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer


# Load the model-ready data created in Notebook 03.
data = pd.read_csv(
    "../data/favorita/store3_item589403_tft_ready.csv"
)

# Convert the date column back to datetime.
#
# CSV files do not preserve Pandas datetime types automatically,
# so pd.read_csv() initially reads the dates as text.
data["date"] = pd.to_datetime(data["date"])


# Inspect the first few rows.
print("Rows:", len(data))
print("Date range:", data["date"].min(), "to", data["date"].max())

data.head()

In [ ]:
# TFT will look at the previous 120 days
# when making each forecast.
max_encoder_length = 120

# TFT will predict up to the next 60 days.
max_prediction_length = 60


# Reserve the final 60 days for testing.
test_length = max_prediction_length

# Reserve the 60 days immediately before test
# for validation.
validation_length = max_prediction_length


# Split chronologically.
#
# iloc works with row positions:
# - last 60 rows -> test
# - 60 rows before that -> validation
# - everything before the last 120 rows -> training
test_df = data.iloc[
    -test_length:
].copy()

validation_df = data.iloc[
    -(test_length + validation_length):-test_length
].copy()

train_df = data.iloc[
    :-(test_length + validation_length)
].copy()


# Verify the new setup.
print("Historical window:", max_encoder_length, "days")
print("Maximum forecast horizon:", max_prediction_length, "days")

print("Training rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))


In [ ]:
# These columns are treated as categorical variables.
categorical_columns = [
    "series_id",
    "day_of_week",
    "month",
    "promotion_status"
]

# Make sure categorical columns are stored as strings.
for column in categorical_columns:
    data[column] = data[column].astype(str)


# Static categorical variable:
# this identifies our single store-item series.
static_categoricals = [
    "series_id"
]


# Future-known categorical variables.
# day_of_week and month are known from the calendar.
# promotion_status is known when the user supplies a planned promotion schedule.
time_varying_known_categoricals = [
    "day_of_week",
    "month",
    "promotion_status"
]


# Future-known numerical variables.
time_varying_known_reals = [
    "time_idx",
    "year",
    "day_of_month"
]


# Future-unknown numerical variable.
# unit_sales is what we want to predict.
time_varying_unknown_reals = [
    "unit_sales"
]


print("Variable groups recreated successfully.")

In [ ]:
# The final training time index.
training_cutoff = train_df["time_idx"].max()

# The final validation time index.
validation_cutoff = validation_df["time_idx"].max()

print("Training cutoff:", training_cutoff)
print("Validation cutoff:", validation_cutoff)

In [ ]:
# Create the training dataset.
# Each training sample uses up to 120 historical days
# to predict the following 60 days.
training_dataset = TimeSeriesDataSet(
    data[data["time_idx"] <= training_cutoff],

    # Time ordering and prediction target.
    time_idx="time_idx",
    target="unit_sales",

    # Identifies our store-item time series.
    group_ids=["series_id"],

    # 120 historical days -> 60 future days.
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    # Tell TFT which variables belong to which category.
    static_categoricals=static_categoricals,
    time_varying_known_categoricals=time_varying_known_categoricals,
    time_varying_known_reals=time_varying_known_reals,
    time_varying_unknown_reals=time_varying_unknown_reals,
)


# Create the validation dataset using the same configuration.
# Its final 60 days form the validation forecast window.
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    data[data["time_idx"] <= validation_cutoff],
    predict=True,
    stop_randomization=True
)


# Create the test dataset using the complete timeline.
# Its final 60 days form the untouched test forecast window.
test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    data,
    predict=True,
    stop_randomization=True
)


# Verify that the datasets were created correctly.
print("Training samples:", len(training_dataset))
print("Validation samples:", len(validation_dataset))
print("Test samples:", len(test_dataset))

In [ ]:
# Number of forecasting samples processed together
# during one training step.
batch_size = 64


# Training loader.
train_dataloader = training_dataset.to_dataloader(
    train=True,
    batch_size=batch_size,
    shuffle=True
)


# Validation loader.
validation_dataloader = validation_dataset.to_dataloader(
    train=False,
    batch_size=batch_size,
    shuffle=False
)


# Test loader.
test_dataloader = test_dataset.to_dataloader(
    train=False,
    batch_size=batch_size,
    shuffle=False
)


print("Training batches:", len(train_dataloader))
print("Validation batches:", len(validation_dataloader))
print("Test batches:", len(test_dataloader))

In [ ]:
# QuantileLoss allows TFT to predict multiple demand quantiles
# instead of producing only a single point forecast.
from pytorch_forecasting.metrics import QuantileLoss

# Lightning manages the neural-network training process.
import lightning.pytorch as pl

# Callbacks used during training.
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

# Fix the random seed so repeated training runs are reproducible.
pl.seed_everything(42, workers=True)

# Create the Temporal Fusion Transformer model.
tft = TemporalFusionTransformer.from_dataset(
    training_dataset,

    # Controls the internal model capacity.
    hidden_size=16,

    # Number of attention heads.
    attention_head_size=4,

    # Helps reduce overfitting during training.
    dropout=0.1,

    # Size of the internal networks used for continuous variables.
    hidden_continuous_size=8,

    # Controls the scale of weight updates during training.
    learning_rate=0.001,

    # Predict multiple quantiles so we can model uncertainty.
    loss=QuantileLoss(),

    # Log training information every 10 batches.
    log_interval=10,

    # No additional validation metrics yet.
    logging_metrics=None,
)

In [ ]:
# Stop training if validation loss has not improved
# for 5 consecutive epochs.
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    mode="min"
)

# Save the checkpoint with the lowest validation loss.
checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="best-tft-60day"
)

# Final trainer for the 120-day history -> 60-day forecast model.
trainer = pl.Trainer(
    max_epochs=30,
    accelerator="auto",
    devices=1
    gradient_clip_val=0.1,
    callbacks=[
        early_stopping,
        checkpoint_callback
    ],
    enable_model_summary=True
)

print("Final 30-epoch Trainer created successfully.")

In [ ]:
# Show the path of the best saved TFT checkpoint.
print("Best checkpoint path:")
print(checkpoint_callback.best_model_path)

# Show the validation loss achieved by that best checkpoint.
print("\nBest validation loss:")
print(checkpoint_callback.best_model_score)

In [16]:
import time

print("Starting final TFT training...")

start_time = time.time()

# Train the final TFT model.
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=validation_dataloader
)

elapsed_time = time.time() - start_time

print("\nFinal TFT training finished.")
print("Time taken:", round(elapsed_time, 2), "seconds")

print("\nBest checkpoint:")
print(checkpoint_callback.best_model_path)

print("\nBest validation loss:")
print(checkpoint_callback.best_model_score)

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Final TFT training finished.
Time taken: 54.12 seconds

Best checkpoint:
c:\Users\karra\Desktop\inventory-forecasting\notebooks\lightning_logs\version_6\checkpoints\best-tft-60day.ckpt

Best validation loss:
tensor(3.6411)


# Temporal Fusion Transformer Training

This notebook trains a Temporal Fusion Transformer (TFT) for up to 60-day
daily demand forecasting using 120 days of historical context.

The model uses historical demand, calendar features, and planned promotion
information to support promotion-aware demand forecasting and downstream
inventory replenishment decisions.

In [18]:
# Load the best saved TFT model from the checkpoint.
#
# best_model_path contains the location of the checkpoint
# that achieved the lowest validation loss during this training run.
best_tft = TemporalFusionTransformer.load_from_checkpoint(
    checkpoint_callback.best_model_path
)

print("Best TFT checkpoint loaded successfully.")

Best TFT checkpoint loaded successfully.
